In order to convert the acceleration data (AD) given in bits into gravity, use this equation: 

Acceleration [g]: [(2*Range)/(2^Resolution)]*AD

In order to convert the rotation data (RD) given in bits into angular velocity, use this equation:

Angular velocity [°/s]: [(2*Range)/(2^Resolution)]*RD


Importamos las librerias.


In [4]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Define la ruta de un archivo (asegúrate de apuntar a tu carpeta de datos)
ruta_archivo = "Archivos caídas/F03_SA03_R03.txt"

# 2. Abre el archivo, elimina los ';' al final de cada fila y carga los números
with open(ruta_archivo, "r") as f:
  lineas_limpias = [linea.replace(";", "") for linea in f]

datos_crudos = np.loadtxt(lineas_limpias, delimiter=",")

# 3. Verificación de dimensiones
print("Forma de la matriz (filas, columnas):", datos_crudos.shape)
print("Primera fila de datos crudos:\n", datos_crudos[0])

Forma de la matriz (filas, columnas): (3000, 9)
Primera fila de datos crudos:
 [   4. -257.  -73.  -39.   71.    6.  -57. -988. -138.]


Las 3 primeras colummnas de los datos crudos, corresponden a los ejes X,Y,Z del acelerometro ADXL345, el cual entrega los datos en bits, con una resolucion de 13 bits y un rango de $\pm 16g$ 
para convertir los datos de aceleracion a unidades de gravedad, usamos la ecuacion: 
Acceleration [g]: [(2*Range)/(2^Resolution)]*AD, dada en el archivo README

Debemos aplicar una norma vectorial a los datos para obtener su magnitud, asi obtenemos una señal unica que no depende de la orientacion del sensor
$$a_{\text{total}}(t) = \sqrt{a_x(t)^2 + a_y(t)^2 + a_z(t)^2}$$

In [15]:
import plotly.graph_objects as go

# 1. Factor de conversión del acelerómetro ADXL345 (según Readme: +-16g, 13 bits)
factor_g = (2 * 16) / (2**13)

# 2. Extraer columnas 0, 1 y 2 (X, Y, Z) y multiplicar por el factor
ax = datos_crudos[:, 0] * factor_g
ay = datos_crudos[:, 1] * factor_g
az = datos_crudos[:, 2] * factor_g

# 3. Calcular la magnitud de la aceleración total en cada instante t
a_total = np.sqrt(ax**2 + ay**2 + az**2)

# 4. Crear el vector de tiempo en segundos (frecuencia de muestreo = 200 Hz)

#Debido a que los archivos no nos dan los datos de los segundos, tenemos que construir el eje de tiempo a partir 
#de la cantidad de muestras y la frecuencia de muestreo. 
#La frecuencia de muestreo es de 200 Hz, es decir, 200 muestras por segundo. 
#Por lo tanto, el tiempo en segundos se puede calcular como el número de muestras dividido por la frecuencia de muestreo.
fs_muestreo = 200
tiempo = np.arange(len(a_total)) / fs_muestreo

# 5. Verificación visual en el dominio del tiempo
fig = go.Figure()
fig.add_trace(go.Scatter(x=tiempo, y=a_total, mode="lines"))
fig.update_layout(
    title="Aceleración Total a_total(t)",
    xaxis_title="Tiempo (s)",
    yaxis_title="Aceleración (g)",
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
    width=900,
    height=400,
)
fig.show()

Ahora necesitamos eliminar la señal constante de la gravedad:
Para eso, primero calculamos el promedio aritmetico de toda la señal calculando un promedio de esta usando la funcion np.mean()
luego en cada muestra individual de a_total, es decir, las magnitudes en diferentes tiempo de la señal, se le resta el valor promedio hallado, que seria aproximadamente 1g correspondiente a la gravedad.

la variable N, guarda la cantidad total de emuestras de la señal (por ejemplo 3000 muestras para un archivo de 15s)

np.fft.rfft(a_sin_dc): Ejecuta el algoritmo de la Transformada Rápida de Fourier para entradas reales (Real Fast Fourier Transform) y guarda el resultado en fft_compleja

np.abs(fft_compleja) / N: Calcula el módulo de cada número complejo resultante y lo divide entre el número total de muestras N. El módulo corresponde al espectro de magnitud $\vert{}F(\omega)\vert{}$ de la Transformada de Fourier. Dividir entre N normaliza la amplitud respecto a la duración del registro, permitiendo comparar en la misma escala archivos cortos de caídas ($15\text{ s}$, $N=3000$) frente a archivos largos de actividades diarias ($100\text{ s}$, $N=20000$).


In [16]:
import plotly.graph_objects as go

# 1. Quitar la componente constante de gravedad (0 Hz / DC)
a_sin_dc = a_total - np.mean(a_total)

# 2. Calcular la FFT para señales reales y su magnitud normalizada
N = len(a_sin_dc)
fft_compleja = np.fft.rfft(a_sin_dc)
magnitud = np.abs(fft_compleja) / N

# 3. Construir el eje X de frecuencias en Hz (sabemos que fs_muestreo = 200 Hz)
frecuencias = np.fft.rfftfreq(N, d=1 / fs_muestreo)

# 4. Graficar el espectro de frecuencias
fig = go.Figure()
fig.add_trace(go.Scatter(x=frecuencias, y=magnitud, mode="lines"))
fig.update_layout(
    title="Espectro de Frecuencias |F(f)| - Caída F03",
    xaxis_title="Frecuencia (Hz)",
    yaxis_title="Magnitud",
    xaxis=dict(range=[0, 50], showgrid=True),  # Hacemos zoom de 0 a 50 Hz donde ocurre el movimiento humano
    yaxis=dict(showgrid=True),
    width=900,
    height=400,
)
fig.show()

Este bloque de codigo, lo que hace en resumen, es volver a aplicar los codigos anteriores, pero a los archivos especificados en el diccionario, superponerlos mediante un plot y compararlos para obtener los fs y fp.
Físicamente, el impacto de una caída contra el suelo se aproxima a un impulso abrupto, por lo que proyecta densidad espectral $\vert{}F(\omega)\vert{}$ hacia frecuencias altas. Por el contrario, las actividades cotidianas como caminar y trotar son movimientos rítmicos/periódicos cuya energía se concentra en frecuencias bajas.

El ciclo for extrae cada pareja clave-valor del diccionario. En cada iteración invoca a obtener_espectro(ruta) y trazala curva mediante plt.plot(..., alpha=0.8), donde alpha controla la opacidad ($80\%$) para que las líneas superpuestasno se oculten entre sí.


En la teoría de Fourier, una señal periódica o repetitiva en el tiempo concentra toda su energía en frecuencias discretas muy estrechas (la frecuencia fundamental y sus armónicos). Trotar (D04) es un movimiento rítmico de 100 segundos donde la persona da aproximadamente 3 zancadas por segundo de manera constante; por eso toda la energía se apila en una torre angosta en $f \approx 3.0\text{ Hz}$ y sus múltiplos ($5.8\text{ Hz}$, $8.7\text{ Hz}$).   Por el contrario, una caída (F05, F06) es un evento aperiódico y abrupto (un impacto de fracción de segundo dentro de un registro de $15\text{ s}$). Por propiedades de la Transformada de Fourier, un pulso corto y brusco en el tiempo esparce su energía a lo largo de un ancho de banda muy amplio (desde $0\text{ Hz}$ hasta casi $30\text{ Hz}$), en lugar de concentrarla en un solo pico alto

El filtro a escoger seria **pasa altas**, debido a que las actividades presentan picos en las frecuencias bajas y disminuyen de forma mucho más abrupta en frecuencias altas, sin embargo, actividades anomalas como las caidas, las cuales no presentan un ritmo, mantiene su frecuencia mejor a lo largo del ancho de banda.

Para la frecuencia de rechazo fs, escogemos un valor de 10Hz, ya que las actividades de trotar y caminar presentan picos por debajo de dicha frecuencia

Para la frecuencia de paso, escogemos 16Hz, ya que a partir de dicha frecuencia, el trote o el caminar ya no mantiene su energia en comparacion con las actividades anomalas como las caidas

In [14]:
import numpy as np
import plotly.graph_objects as go


# 1. Función reutilizable que recibe la ruta de UN archivo y devuelve su FFT
def obtener_espectro(ruta):
    # Leer y limpiar los ';'
    with open(ruta, "r") as f:
        lineas = [linea.replace(";", "") for linea in f]
    datos = np.loadtxt(lineas, delimiter=",")

    # Convertir a g (ADXL345: 13 bits, +-16g) y calcular magnitud total[cite: 1, 2]
    factor_g = (2 * 16) / (2**13)
    ax = datos[:, 0] * factor_g
    ay = datos[:, 1] * factor_g
    az = datos[:, 2] * factor_g
    a_total = np.sqrt(ax**2 + ay**2 + az**2)

    # Quitar gravedad constante (0 Hz) y calcular FFT normalizada
    a_sin_dc = a_total - np.mean(a_total)
    N = len(a_sin_dc)
    magnitud = np.abs(np.fft.rfft(a_sin_dc)) / N
    frecuencias = np.fft.rfftfreq(N, d=1 / 200)

    return frecuencias, magnitud


# 2. Diccionario con los 4 archivos clave de la Fase 1 (ajusta las rutas a tus carpetas)
archivos = {
    "Caminar lento (D01)": r"Archivos caídas/D01_SA01_R01.txt",
    "Trotar rápido (D04)": r"Archivos caídas/D04_SA03_R01.txt",
    "Caída 1 (F05)": r"Archivos caídas/F05_SA03_R01.txt",
    "Caída 2 (F06)": r"Archivos caídas/F06_SA03_R01.txt",
}

# 3. Crear la figura interactiva en Plotly y agregar las 4 curvas superpuestas
fig = go.Figure()

for etiqueta, ruta in archivos.items():
    freqs, mag = obtener_espectro(ruta)
    fig.add_trace(
        go.Scatter(
            x=freqs,
            y=mag,
            mode="lines",
            name=etiqueta,
            opacity=0.8,
        )
    )

# 4. Configurar títulos, ejes y zoom entre 0 y 40 Hz
fig.update_layout(
    title="Comparación Espectral: Actividades Diarias vs. Caídas",
    xaxis_title="Frecuencia (Hz)",
    yaxis_title="Magnitud |F(f)|",
    xaxis=dict(range=[0, 40], showgrid=True),
    yaxis=dict(showgrid=True),
    template="plotly_white",
    width=1000,
    height=450,
    hovermode="x unified",
)

fig.show()

### Importación de archivos como dataframes

In [8]:
import pandas as pd
from pathlib import Path

# Carpeta con los .txt (relativa a la carpeta del notebook/script: Proyecto_SisFall)
CARPETA = Path("Archivos caídas")

COLUMNAS = [
    'ADXL_X', 'ADXL_Y', 'ADXL_Z',  # Acelerómetro 1
    'ITG_X', 'ITG_Y', 'ITG_Z',     # Giroscopio
    'MMA_X', 'MMA_Y', 'MMA_Z'      # Acelerómetro 2
]

# Factores de conversión: (2 * Rango) / (2^Resolución)
FACTORES = {
    'ADXL': (2 * 16) / 2**13,    # ±16 g, 13 bits
    'ITG':  (2 * 2000) / 2**16,  # ±2000 °/s, 16 bits
    'MMA':  (2 * 8) / 2**14,     # ±8 g, 14 bits
}

def importar_y_limpiar(nombre_archivo, carpeta=CARPETA):
    """
    Importa un .txt de SisFall, lo limpia y lo convierte a unidades físicas.
    - nombre_archivo: con o sin extensión, ej. 'D04_SA01_R01' o 'D04_SA01_R01.txt'
    - Devuelve el DataFrame y además lo deja en memoria como df_<nombre>.
    """
    nombre = Path(nombre_archivo).stem
    ruta = Path(carpeta) / f"{nombre}.txt"

    # 1. Lectura
    df = pd.read_csv(ruta, sep=',', engine='python', header=None, names=COLUMNAS)

    # 2. Limpieza (el último dato de cada fila trae ';')
    df['MMA_Z'] = df['MMA_Z'].astype(str).str.replace(';', '').str.strip().astype(int)
    df = df.astype(int)

    # 3. Conversión de bits a valores físicos
    for prefijo, factor in FACTORES.items():
        cols = [f'{prefijo}_X', f'{prefijo}_Y', f'{prefijo}_Z']
        df[cols] = df[cols] * factor

    # 4. Guardar en memoria con el nombre df_<archivo>
    globals()[f"df_{nombre}"] = df
    print(f"✔ '{nombre}.txt' cargado → df_{nombre} ({df.shape[0]} filas)")
    return df

In [9]:
# Ejemplo de uso:

df_D04_SA01_R01 = importar_y_limpiar("D04_SA01_R01")
df_D04_SA01_R01.head()

✔ 'D04_SA01_R01.txt' cargado → df_D04_SA01_R01 (20000 filas)


,ADXL_X,ADXL_Y,ADXL_Z,ITG_X,ITG_Y,ITG_Z,MMA_X,MMA_Y,MMA_Z
0,0.375000,-1.605469,-0.410156,24.047852,30.761719,58.227539,0.344727,-1.586914,-0.291992
1,0.304688,-1.613281,-0.343750,5.737305,32.104492,61.096191,0.271484,-1.590820,-0.223633
2,0.218750,-1.671875,-0.238281,-7.751465,34.179688,64.758301,0.203125,-1.635742,-0.142578
3,0.089844,-1.691406,-0.128906,-14.587402,35.827637,64.514160,0.087891,-1.631836,-0.011719
4,-0.042969,-1.691406,0.003906,-14.465332,34.973145,63.476562,-0.026367,-1.631836,0.120117


### Función Graficar en Tiempo

In [10]:
import numpy as np
import plotly.graph_objects as go

FS = 200  # Frecuencia de muestreo (Hz)


def graficar_serie_tiempo(df, columna, actividad=None, fs=FS, color='royalblue', devolver=False):
    if columna not in df.columns:
        raise ValueError(f"'{columna}' no existe. Columnas disponibles: {list(df.columns)}")

    tiempo = np.arange(len(df)) / fs
    unidad_y = "Velocidad Angular (°/s)" if columna.startswith("ITG") else "Aceleración (g)"

    titulo = f"Serie de Tiempo: Sensor {columna}"
    if actividad:
        titulo += f" - {actividad}"

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=tiempo,
        y=df[columna],
        mode='lines',
        name=columna,
        line=dict(width=2, color=color)
    ))

    fig.update_layout(
        title=titulo,
        xaxis_title="Tiempo (Segundos)",
        yaxis_title=unidad_y,
        template="plotly_white",
        hovermode="x unified"
    )

    fig.show()
    if devolver:
        return fig

In [11]:
graficar_serie_tiempo(df_D04_SA01_R01, 'ADXL_Y', actividad="Persona trotando suave")